# UK Social Housing Implicit Subsidy Analysis

This notebook loads the processed data and produces exploratory visualisations of the implicit subsidy received by social and affordable rent tenants across English local authorities.

**The implicit subsidy** = market rent − social/affordable rent. It is the economic value of below-market housing that never appears in a government budget line.

Data sources:
- **Market rents**: ONS Private Rental Market Statistics (Oct 2022–Sep 2023)
- **Social & affordable rents**: RSH Statistical Data Return 2024–25 (March 2025)
- **Stock**: MHCLG Live Table 100 (March 2024)

See `README.md` for full methodology and caveats.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import geopandas as gpd
from pathlib import Path

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PROCESSED = Path('../data/processed')
RAW = Path('../data/raw')

summary = pd.read_csv(PROCESSED / 'subsidy_summary_by_la.csv')
long    = pd.read_csv(PROCESSED / 'subsidy_by_la_bedroom.csv')

print(f'Summary: {len(summary)} LAs, {summary["region"].nunique()} regions')
print(f'Long:    {len(long)} rows (LA × bedroom)')
summary.head(3)

## 1. National overview

In [ ]:
total_social_bn   = summary['total_annual_subsidy_social'].sum() / 1e9
total_affrd_bn    = summary['total_annual_subsidy_affordable'].sum() / 1e9
median_soc_yr     = summary['subsidy_social_wtavg_annual'].median()
median_affrd_yr   = summary['subsidy_affordable_wtavg_annual'].median()
n_las             = summary['la_code'].nunique()

print(f"=== National totals (England) ===")
print(f"LAs covered:                    {n_las}")
print(f"Total implicit social subsidy:  £{total_social_bn:.1f}bn / year")
print(f"Total implicit afford. subsidy: £{total_affrd_bn:.1f}bn / year")
print(f"Median social subsidy per unit: £{median_soc_yr:,.0f} / year")
print(f"Median afford. subsidy per unit:£{median_affrd_yr:,.0f} / year")

## 2. Distribution of social subsidy across LAs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Social subsidy distribution
ax = axes[0]
data = summary['subsidy_social_wtavg_annual'].dropna()
ax.hist(data, bins=40, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: £{data.median():,.0f}')
ax.set_xlabel('Stock-weighted social subsidy (£/unit/year)')
ax.set_ylabel('Number of LAs')
ax.set_title('Distribution of implicit social subsidy')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()

# Affordable subsidy distribution
ax = axes[1]
data2 = summary['subsidy_affordable_wtavg_annual'].dropna()
ax.hist(data2, bins=40, color='darkorange', edgecolor='white', linewidth=0.5)
ax.axvline(data2.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: £{data2.median():,.0f}')
ax.set_xlabel('Stock-weighted affordable subsidy (£/unit/year)')
ax.set_ylabel('Number of LAs')
ax.set_title('Distribution of implicit affordable subsidy')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
ax.legend()

plt.tight_layout()
plt.savefig(PROCESSED / 'fig_distribution.png', bbox_inches='tight')
plt.show()

## 3. Top and bottom 20 LAs — social subsidy per unit

In [ ]:
def la_bar_chart(df, col, title, color, n=20, ax=None):
    sub = df[['la_name', col]].dropna().sort_values(col)
    top = sub.tail(n)
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    bars = ax.barh(top['la_name'], top[col], color=color, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('£/unit/year')
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
    ax.tick_params(axis='y', labelsize=8)
    return ax

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

la_bar_chart(summary, 'subsidy_social_wtavg_annual',
             'Top 20 LAs — social subsidy/unit/yr', 'steelblue', ax=axes[0])
la_bar_chart(summary, 'subsidy_affordable_wtavg_annual',
             'Top 20 LAs — affordable subsidy/unit/yr', 'darkorange', ax=axes[1])

plt.tight_layout()
plt.savefig(PROCESSED / 'fig_top20_per_unit.png', bbox_inches='tight')
plt.show()

In [ ]:
# Bottom 20 (lowest subsidy — where affordable / market rents converge)
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

def la_bar_bottom(df, col, title, color, n=20, ax=None):
    sub = df[['la_name', col]].dropna().sort_values(col).head(n).sort_values(col, ascending=False)
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 7))
    ax.barh(sub['la_name'], sub[col], color=color, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('£/unit/year')
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
    ax.tick_params(axis='y', labelsize=8)
    return ax

la_bar_bottom(summary, 'subsidy_social_wtavg_annual',
              'Bottom 20 LAs — social subsidy/unit/yr', 'steelblue', ax=axes[0])
la_bar_bottom(summary, 'subsidy_affordable_wtavg_annual',
              'Bottom 20 LAs — affordable subsidy/unit/yr', 'darkorange', ax=axes[1])

plt.tight_layout()
plt.savefig(PROCESSED / 'fig_bottom20_per_unit.png', bbox_inches='tight')
plt.show()

## 4. Top 20 LAs by total annual subsidy bill

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, col, color, label in [
    (axes[0], 'total_annual_subsidy_social',     'steelblue',  'Social'),
    (axes[1], 'total_annual_subsidy_affordable',  'darkorange', 'Affordable'),
]:
    sub = summary[['la_name', col]].dropna().sort_values(col).tail(20)
    ax.barh(sub['la_name'], sub[col] / 1e6, color=color, edgecolor='white', linewidth=0.3)
    ax.set_xlabel('£ million / year')
    ax.set_title(f'Top 20 LAs — total annual {label.lower()} subsidy bill')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig(PROCESSED / 'fig_top20_total_bill.png', bbox_inches='tight')
plt.show()

## 5. Regional comparison

In [ ]:
region_order = (
    summary.groupby('region')['subsidy_social_wtavg_annual']
    .median().sort_values(ascending=False).index.tolist()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, color, label in [
    (axes[0], 'subsidy_social_wtavg_annual',     'steelblue',  'social'),
    (axes[1], 'subsidy_affordable_wtavg_annual',  'darkorange', 'affordable'),
]:
    plot_data = summary.dropna(subset=[col, 'region'])
    sns.boxplot(
        data=plot_data,
        y='region', x=col,
        order=region_order,
        color=color,
        ax=ax,
        linewidth=0.8,
        fliersize=3,
    )
    ax.set_xlabel(f'Annual {label} subsidy (£/unit)')
    ax.set_ylabel('')
    ax.set_title(f'Implicit {label} subsidy by region')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
    ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(PROCESSED / 'fig_regional_boxplot.png', bbox_inches='tight')
plt.show()

## 6. Subsidy by bedroom size

In [ ]:
bed_labels = {'1_bed': '1 bed', '2_bed': '2 bed', '3_bed': '3 bed', '4plus_bed': '4+ bed'}
long['bedroom_label'] = long['bedrooms'].map(bed_labels)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, color, label in [
    (axes[0], 'subsidy_social_monthly',     'steelblue',  'social'),
    (axes[1], 'subsidy_affordable_monthly',  'darkorange', 'affordable'),
]:
    plot_df = long.dropna(subset=[col])
    sns.boxplot(
        data=plot_df,
        x='bedroom_label', y=col,
        order=['1 bed', '2 bed', '3 bed', '4+ bed'],
        color=color, ax=ax,
        linewidth=0.8, fliersize=2,
    )
    ax.set_xlabel('Bedroom size')
    ax.set_ylabel(f'Monthly {label} subsidy (£/unit)')
    ax.set_title(f'Implicit {label} subsidy by bedroom size (all LAs)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))

plt.tight_layout()
plt.savefig(PROCESSED / 'fig_bedroom_boxplot.png', bbox_inches='tight')
plt.show()

## 7. Market vs social rent scatter

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

bed_order = ['1_bed', '2_bed', '3_bed', '4plus_bed']
bed_display = {'1_bed': '1 bedroom', '2_bed': '2 bedrooms',
               '3_bed': '3 bedrooms', '4plus_bed': '4+ bedrooms'}

for i, bed in enumerate(bed_order):
    sub = long[long['bedrooms'] == bed].dropna(subset=['market_rent_monthly', 'social_rent_monthly'])
    ax = axes[i]
    ax.scatter(sub['market_rent_monthly'], sub['social_rent_monthly'],
               alpha=0.6, s=20, color='steelblue')
    mx = sub[['market_rent_monthly', 'social_rent_monthly']].max().max()
    ax.plot([0, mx], [0, mx], 'k--', linewidth=0.8, label='market = social')
    ax.set_xlabel('Market rent (£/month)')
    ax.set_ylabel('Social rent (£/month)')
    ax.set_title(bed_display[bed])
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{int(x):,}'))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{int(x):,}'))

fig.suptitle('Market rent vs social rent by LA and bedroom size', y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED / 'fig_market_vs_social_scatter.png', bbox_inches='tight')
plt.show()

## 8. Choropleth map — social subsidy per unit

In [ ]:
# Load boundary GeoJSON (2013 LA codes; most still match current ONS codes)
gdf = gpd.read_file(RAW / 'england_lad_boundaries.geojson')
print(f'Boundary file: {len(gdf)} features, CRS: {gdf.crs}')
print('Code column:', gdf.columns.tolist()[:5])

# Merge with summary data — join on la_code / LAD13CD
map_data = gdf.merge(
    summary[['la_code', 'subsidy_social_wtavg_annual', 'subsidy_affordable_wtavg_annual']],
    left_on='LAD13CD', right_on='la_code',
    how='left'
)
matched = map_data['la_code'].notna().sum()
print(f'Matched {matched}/{len(gdf)} LAs to subsidy data')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 10))

for ax, col, title, cmap in [
    (axes[0], 'subsidy_social_wtavg_annual',     'Social subsidy\n(£/unit/year)',     'YlOrRd'),
    (axes[1], 'subsidy_affordable_wtavg_annual',  'Affordable subsidy\n(£/unit/year)', 'YlOrBr'),
]:
    map_data.plot(
        column=col,
        ax=ax,
        cmap=cmap,
        legend=True,
        legend_kwds={'label': title, 'orientation': 'vertical', 'shrink': 0.6},
        missing_kwds={'color': 'lightgrey', 'label': 'No data'},
        edgecolor='white',
        linewidth=0.2,
    )
    ax.set_axis_off()
    ax.set_title(title.replace('\n', ' '), fontsize=12)

fig.suptitle('Implicit housing subsidy by local authority (England)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED / 'fig_choropleth.png', bbox_inches='tight', dpi=200)
plt.show()

## 9. Data quality — matching and coverage notes

In [ ]:
print('=== Coverage summary ===')
print(f'LAs in summary:          {len(summary)}')
print(f'LAs with social subsidy: {summary["subsidy_social_wtavg_annual"].notna().sum()}')
print(f'LAs with afford. subsidy:{summary["subsidy_affordable_wtavg_annual"].notna().sum()}')
print(f'LAs with missing region: {summary["region"].isna().sum()}')
print()
print('=== LAs missing social subsidy (no ONS or no RSH data) ===')
missing = summary[summary['subsidy_social_wtavg_annual'].isna()][['la_name','la_code','region']]
print(missing.to_string(index=False))

In [ ]:
# Negative affordable subsidies: where affordable rent > market rent
neg_aff = summary[summary['subsidy_affordable_wtavg_annual'].notna() &
                  (summary['subsidy_affordable_wtavg_annual'] < 0)]
print(f'LAs where affordable rent > market rent: {len(neg_aff)}')
print(neg_aff[['la_name', 'region', 'subsidy_affordable_wtavg_annual']]
      .sort_values('subsidy_affordable_wtavg_annual')
      .head(10).to_string(index=False))